<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 04. Gradient Boosting y Clases Desbalanceadas
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Programación para Ciencia de Datos
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 09
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Data%20Science%20programming/09%20-%20Decision%20Trees/Para%20Dummies/04_Boosting_y_Casos_Estudio_Desbalanceados_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🚀

En el cuaderno anterior combinamos árboles que se entrenaban **al mismo tiempo, de forma independiente** (Random Forest). Aquí veremos una estrategia distinta: entrenar árboles **uno después de otro**, donde cada nuevo árbol se enfoca en corregir los errores del anterior. También veremos qué hacer cuando una de las categorías que queremos predecir es mucho más rara que la otra (por ejemplo, detectar fraude o una enfermedad poco común).

Al terminar podrás explicar, con tus propias palabras:
1. La diferencia entre entrenar árboles "en paralelo" (Bagging) y "en cadena" (Boosting).
2. Por qué Boosting suele usar árboles muy pequeños y débiles como piezas base.
3. Qué significa que un dataset esté "desbalanceado" y por qué eso engaña a los modelos.
4. Una forma sencilla de corregir ese desbalance (`class_weight='balanced'`).

---
## 1. Aprender de los errores, examen tras examen 📝

Imagina un estudiante que presenta el mismo examen de práctica varias veces seguidas. Después de la primera vuelta, revisa cuáles preguntas falló y, en la segunda vuelta, se enfoca sobre todo en esas — sin olvidar del todo el resto. Repite el proceso muchas veces, y con cada vuelta se concentra más en lo que todavía le cuesta.

Así funciona el **Boosting**: entrena árboles **uno después de otro** (no todos a la vez, como en Bagging). Cada árbol nuevo presta más atención a los ejemplos que los árboles anteriores clasificaron mal. Los árboles usados suelen ser muy simples y "débiles" (a veces de una sola pregunta), pero al sumarlos todos en cadena se vuelven un equipo muy preciso.

`GradientBoostingClassifier` es la versión de Scikit-Learn de esta idea. Vamos a probarlo en un caso típico: detectar fraude, donde los casos de fraude son mucho más raros que los legítimos.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report

# Datos simulados: 95% transacciones legítimas, 5% fraude
X, y = make_classification(
    n_samples=2000, weights=[0.95, 0.05], random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

modelo_boosting = GradientBoostingClassifier(
    n_estimators=100, learning_rate=0.1, random_state=42
).fit(X_train, y_train)

y_pred = modelo_boosting.predict(X_test)
print(classification_report(y_test, y_pred, target_names=['Legítimo (95%)', 'Fraude (5%)']))

### 🤔 ¿Qué acaba de pasar?

- `make_classification(weights=[0.95, 0.05])` crea datos falsos a propósito desbalanceados: solo el 5% de los casos son "fraude".
- Entrenamos un `GradientBoostingClassifier`, que va sumando árboles en cadena, cada uno corrigiendo un poco los errores del anterior.
- `classification_report` nos muestra, por separado, qué tan bien detecta el modelo cada categoría. Fíjate en la fila de "Fraude": aunque la exactitud general (accuracy) suele verse alta, el **recall** (qué proporción de los fraudes reales detectó) suele ser más bajo — el modelo tiende a "ignorar" un poco a la clase minoritaria porque acertando siempre en la mayoritaria ya obtiene buena exactitud.

---
## 2. La aguja en el pajar: cuando una categoría casi no aparece ⚖️

Si el 95% de tus datos son "legítimos", un modelo perezoso podría predecir siempre "legítimo" y aun así acertar el 95% de las veces — pero sería completamente inútil para detectar fraude, que es justo lo que nos interesa.

Una solución sencilla es decirle al modelo: "los errores en la categoría rara valen más caro". En Scikit-Learn esto se hace con el parámetro `class_weight='balanced'`, que ajusta automáticamente el peso de cada clase según qué tan poco aparece: entre más rara la clase, más "le duele" al modelo equivocarse con ella.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score

modelo_normal = RandomForestClassifier(random_state=42).fit(X_train, y_train)
modelo_balanceado = RandomForestClassifier(class_weight='balanced', random_state=42).fit(X_train, y_train)

recall_normal = recall_score(y_test, modelo_normal.predict(X_test))
recall_balanceado = recall_score(y_test, modelo_balanceado.predict(X_test))

print(f'Recall de Fraude SIN balancear:                {recall_normal*100:.2f}%')
print(f'Recall de Fraude CON class_weight="balanced":  {recall_balanceado*100:.2f}%')

### 🤔 ¿Qué acaba de pasar?

- El **recall** de la clase "Fraude" responde a la pregunta: de todos los fraudes reales que había, ¿cuántos detectó el modelo?
- `class_weight='balanced'` le dice al Random Forest que penalice más fuerte los errores sobre la clase minoritaria (fraude) al construir cada árbol.
- Es común ver que el recall de la clase rara mejora al usar `class_weight='balanced'`, aunque a veces a cambio de cometer más falsas alarmas sobre la clase mayoritaria — en problemas como detección de fraude, eso suele ser un intercambio razonable: preferimos revisar algunas transacciones de más antes que dejar pasar un fraude real.

---
##### 🎯 Reto Práctico para Dummies: Compara todo el equipo

Entrena un `RandomForestClassifier` y un `GradientBoostingClassifier`, y compáralos con la exactitud (`.score`) y el recall de la clase Fraude (`recall_score`). ¿Cuál modelo detecta más fraudes reales?

In [ ]:
# =========================================================================
# TU SOLUCIÓN: Reto Dummies 4 - Comparando modelos en datos desbalanceados
# =========================================================================

# modelos = {
#     'Random Forest': RandomForestClassifier(random_state=42),
#     'Gradient Boosting': GradientBoostingClassifier(random_state=42),
# }
# for nombre, modelo in modelos.items():
#     ...


<details>
<summary><b>💡 Haz clic aquí para ver la solución explicada...</b></summary>

```python
modelos = {
    'Random Forest': RandomForestClassifier(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
}

for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    exactitud = modelo.score(X_test, y_test)
    recall_fraude = recall_score(y_test, y_pred)
    print(f'{nombre:>18}: Exactitud = {exactitud*100:.2f}% | Recall Fraude = {recall_fraude*100:.2f}%')
```
</details>

---
## 3. Resumen relámpago ⚡

| Idea | En una frase |
|---|---|
| Boosting | Entrenar árboles uno tras otro, cada uno corrigiendo los errores del anterior. |
| Bagging vs Boosting | Bagging entrena árboles en paralelo para reducir varianza; Boosting entrena en cadena para reducir sesgo. |
| Datos desbalanceados | Cuando una categoría es mucho más rara que la otra (ej. fraude, enfermedades raras). |
| Recall de la clase rara | Qué proporción de los casos reales de la categoría minoritaria detectó el modelo. |
| `class_weight='balanced'` | Le dice al modelo que penalice más los errores sobre la clase minoritaria. |

Con esto cerramos el módulo 09 de Árboles de Decisión: desde una sola pregunta de Sí/No hasta comités enteros de árboles trabajando en equipo.

➡️ **Siguiente paso:** en el cuaderno [00 - Introducción al Clustering (Para Dummies)](../../10%20-%20Clustering/Para%20Dummies/00_Introduccion_al_Clustering_Dummies.ipynb) del módulo 10 dejaremos atrás el aprendizaje supervisado (donde conocíamos las respuestas correctas) para explorar cómo encontrar grupos ocultos en los datos sin etiquetas.

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Programación para Ciencia de Datos (Edición Para No Ingenieros)</i>
  </p>
</div>
